In [ ]:
#| default_exp model

In [ ]:
#| export
from __future__ import annotations

import hashlib
import importlib
import os
from collections.abc import Callable
from pathlib import Path

import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin, hf_hub_download, list_repo_files

In [ ]:
#| include: false
from nbdev.showdoc import *

## Overview

An optimized model no longer has the shapes its factory builds: pruning changes channel counts, and those
counts are the only thing standing between a checkpoint and a model that loads. `FasterModel` publishes
both — the factory that builds the architecture, and the **rebuild spec** of the layers whose shapes changed.

| Piece | What it is | Where it goes |
|---|---|---|
| `source` | dotted path of the factory, `torchvision.models.resnet18` | `config.json` |
| `source_kwargs` | what the factory is called with, e.g. `{'num_classes': 10, 'weights': None}` | `config.json` |
| `modules` | constructor arguments of every shape-carrying leaf, from `spec_from` | `config.json` |
| weights | the optimized weights, under the `net.` prefix | `model.safetensors` |

`save_pretrained` / `from_pretrained` come from `PyTorchModelHubMixin`, so a directory or a Hub repo id
work the same way. Loading is always **strict**: a spec that disagrees with the weights raises instead of
loading in part. `wrap` and `from_pretrained` both return the model in **eval mode**, so a forward pass
reads the published BatchNorm statistics instead of the statistics of the batch it was given.

In [ ]:
#| export
LAYER_TYPES = {'Conv2d': nn.Conv2d, 'Linear': nn.Linear, 'BatchNorm2d': nn.BatchNorm2d, 'BatchNorm1d': nn.BatchNorm1d}

_BN_ARGS = ('num_features', 'eps', 'momentum', 'affine', 'track_running_stats')
_ARGS = {'Conv2d': ('in_channels', 'out_channels', 'kernel_size', 'stride', 'padding', 'dilation', 'groups', 'padding_mode'),
         'Linear': ('in_features', 'out_features'),
         'BatchNorm2d': _BN_ARGS,
         'BatchNorm1d': _BN_ARGS}

In [ ]:
#| export
def resolve(
    source: str,  # dotted path of a model factory, e.g. 'torchvision.models.resnet18'
) -> Callable:
    "Import the model factory `source` points to"
    module, _, attr = source.rpartition('.')
    try:
        return getattr(importlib.import_module(module), attr)
    except (ImportError, AttributeError, ValueError) as e:
        raise ImportError(f"cannot resolve source '{source}' — pass a dotted path to a model factory, "
                          f"e.g. 'torchvision.models.resnet18' ({e})") from e

In [ ]:
show_doc(resolve)

In [ ]:
#| export
def _plain(v):
    "JSON-native value (torch keeps sizes as tuples)"
    return list(v) if isinstance(v, tuple) else v


def spec_from(
    model: nn.Module,  # model whose optimized layers are recorded
) -> dict:
    "Rebuild spec: the constructor arguments of every Conv2d, Linear and BatchNorm leaf, read off the live model"
    spec = {}
    for name, m in model.named_modules():
        kind = type(m).__name__
        if LAYER_TYPES.get(kind) is not type(m): continue   # exact type: a subclass would not rebuild from these arguments
        spec[name] = {'type': kind, **{a: _plain(getattr(m, a)) for a in _ARGS[kind]}}
        if kind in ('Conv2d', 'Linear'): spec[name]['bias'] = m.bias is not None
    return spec

In [ ]:
show_doc(spec_from)

In [ ]:
#| export
def state_hash(
    obj: nn.Module | dict,  # a model or a state dict
) -> str:
    "sha256 over every state dict entry: key, shape, dtype and raw bytes"
    sd = obj.state_dict() if isinstance(obj, nn.Module) else obj
    h = hashlib.sha256()
    for k in sorted(sd):
        t = sd[k]
        h.update(f"{k}{tuple(t.shape)}{t.dtype}".encode())
        h.update(t.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()

In [ ]:
show_doc(state_hash)

In [ ]:
#| export
def _swap(net, name, kw):
    "Replace the leaf at `name` by a module rebuilt from its spec"
    parent, _, child = name.rpartition('.')
    try:
        host = net.get_submodule(parent)
        getattr(host, child)
    except AttributeError as e:
        raise KeyError(f"module '{name}' is not in the source model — check `modules` against the source factory") from e
    kw = dict(kw)
    setattr(host, child, LAYER_TYPES[kw.pop('type')](**kw))


class FasterModel(nn.Module, PyTorchModelHubMixin,
                  library_name='fastermodels',
                  repo_url='https://github.com/FasterAI-Labs/fastermodels',
                  tags=['fasterai']):
    "A model optimized by fasterai: a source factory, the rebuild spec of its layers, and its weights"

    def __init__(self,
                 source: str,                        # dotted path of the model factory
                 source_kwargs: dict | None = None,  # arguments the factory is called with
                 modules: dict | None = None,        # rebuild spec, as returned by `spec_from`
                 recipe: dict | None = None,         # the steps that produced the weights
                 provenance: dict | None = None,     # versions, commits, weights the source started from
    ):
        super().__init__()
        self.net = resolve(source)(**(source_kwargs or {}))
        for name, kw in (modules or {}).items(): _swap(self.net, name, kw)
        self.recipe, self.provenance = recipe, provenance   # these arguments are what the mixin writes to config.json

    def forward(self, x): return self.net(x)

    @classmethod
    def wrap(cls,
             model: nn.Module,                   # the optimized model to publish
             source: str,                        # dotted path of the model factory it came from
             source_kwargs: dict | None = None,  # arguments the factory is called with
             recipe: dict | None = None,         # the steps that produced the weights
             provenance: dict | None = None,     # versions, commits, weights the source started from
    ) -> "FasterModel":
        "Wrap an optimized model, reading its rebuild spec off its layers; returned in eval mode"
        fm = cls(source=source, source_kwargs=source_kwargs, modules=spec_from(model), recipe=recipe, provenance=provenance)
        fm.net.load_state_dict(model.state_dict(), strict=True)
        return fm.eval()

    @classmethod
    def _from_pretrained(cls, **kwargs):
        "Load strict, so a spec that disagrees with the weights raises instead of loading in part, and in eval mode"
        return super()._from_pretrained(**{**kwargs, 'strict': True}).eval()

In [ ]:
show_doc(FasterModel)

In [ ]:
show_doc(FasterModel.wrap)

In [ ]:
#| export
_FORMS = {'safetensors': 'model.safetensors', 'torchscript': 'model.torchscript.pt', 'onnx': 'model.onnx'}


def _fetch(repo_id, name):
    "The local path of `name`, downloaded from the Hub unless `repo_id` is already a directory"
    d = Path(repo_id)
    return d / name if d.is_dir() else Path(hf_hub_download(repo_id, name))


def load(
    repo_id: str,        # a Hub repo id, or a local artifact directory
    form: str = 'auto',  # 'auto', or one of 'safetensors', 'torchscript', 'onnx'
    **kw,                # passed to `FasterModel.from_pretrained`
):
    "Load the form a repo publishes: a FasterModel, a TorchScript module, or the path of the ONNX file"
    if form != 'auto' and form not in _FORMS:
        raise ValueError(f"form must be 'auto' or one of {', '.join(_FORMS)}, got {form!r}")
    d = Path(repo_id)
    files = os.listdir(d) if d.is_dir() else list_repo_files(repo_id)
    for f in (_FORMS if form == 'auto' else [form]):
        if _FORMS[f] not in files: continue
        if f == 'safetensors': return FasterModel.from_pretrained(repo_id, **kw)
        if f == 'torchscript': return torch.jit.load(_fetch(repo_id, _FORMS[f]), map_location='cpu').eval()
        return _fetch(repo_id, _FORMS[f])   # the caller opens the ONNX with the runtime of their choice
    raise FileNotFoundError(f"{repo_id} publishes none of {', '.join(_FORMS.values())}")

In [ ]:
show_doc(load)

---

## Usage

Publish a model you just optimized:

```python
from fastermodels import FasterModel

fm = FasterModel.wrap(pruned, 'torchvision.models.resnet18', {'num_classes': 10, 'weights': None},
                      recipe={'prune': 'ratio 0.3, local, round_to 8'})
fm.save_pretrained('artifact')            # config.json + model.safetensors
fm.push_to_hub('FasterAI-Labs/my-model')  # the same two files, on the Hub
```

Load back whatever a repo publishes, from a directory or from the Hub:

```python
from fastermodels import load

model = load('FasterAI-Labs/my-model')    # safetensors, else TorchScript, else the ONNX path
load('artifact', form='torchscript')      # or name the form you want
```

A variant published in INT8 only has no `model.safetensors`, so `load` looks for `model.safetensors`, then
`model.torchscript.pt`, then `model.onnx`, and raises naming the three when a repo has none of them. The
first two come back ready to call; the ONNX comes back as a path, for the runtime of your choice.

When the repo does publish weights, `FasterModel` reads them directly:

```python
fm = FasterModel.from_pretrained('artifact')
fm(x)                                     # delegates to fm.net, in eval mode
fm.recipe                                 # what produced these weights
state_hash(fm)                            # the same digest the producer published
```

`source_kwargs` never carries pretrained weights (`'weights': None`): loading an artifact downloads nothing
but the artifact itself, and the weights the optimization started from are recorded in `provenance`.

---

## See Also

- [Eval](01_eval.html) - the per-image harness the published numbers come from
- [Card](02_card.html) - what the artifact says about itself
- [Gate](03_gate.html) - the conditions an artifact meets before publication

Tests live in `nbs/tests/test_model.ipynb`.